In [14]:
print("--- Students Table ---")
display(pd.read_sql_query("SELECT * FROM students", conn))

print("\n--- Learning Environment Table ---")
display(pd.read_sql_query("SELECT * FROM learning_environment", conn))

--- Students Table ---


,student_id,gender,age,education_level,adaptivity_level,financial_condition
0,1,Male,20,University,Moderate,Mid
1,2,Female,15,School,Low,Poor
2,3,Male,22,University,High,Rich
3,4,Female,17,School,Low,Mid
4,5,Male,19,University,Moderate,Poor



--- Learning Environment Table ---


,student_id,internet_type,network_type,device,class_duration,location
0,1,Wifi,4G,Laptop,2.0,Yes
1,2,Mobile Data,3G,Phone,1.0,No
2,3,Wifi,4G,Tablet,3.0,Yes
3,4,Mobile Data,4G,Phone,1.5,Yes
4,5,Wifi,3G,Laptop,2.0,No


In [17]:
# Q1: Write a query to display gender, age, education level, and internet type for all students.
q1 = """
SELECT s.gender, s.age, s.education_level, l.internet_type
FROM students s
INNER JOIN learning_environment l ON s.student_id = l.student_id;
"""
pd.read_sql_query(q1, conn)

,gender,age,education_level,internet_type
0,Male,20,University,Wifi
1,Female,15,School,Mobile Data
2,Male,22,University,Wifi
3,Female,17,School,Mobile Data
4,Male,19,University,Wifi


In [18]:
# Q2: Display all students who use Wifi and have Moderate adaptivity level. 
# Show gender, education level, and device.

q2 = """
SELECT s.gender, s.education_level, l.device
FROM students s
INNER JOIN learning_environment l ON s.student_id = l.student_id
WHERE l.internet_type = 'Wifi' AND s.adaptivity_level = 'Moderate';
"""

pd.read_sql_query(q2, conn)

,gender,education_level,device
0,Male,University,Laptop
1,Male,University,Laptop


In [19]:
# Q3: Show all school students who use Mobile Data and have Low adaptivity level. 
# Display gender, age, and network type.

q3 = """
SELECT s.gender, s.age, l.network_type
FROM students s
INNER JOIN learning_environment l ON s.student_id = l.student_id
WHERE s.education_level = 'School' AND l.internet_type = 'Mobile Data' AND s.adaptivity_level = 'Low';
"""
pd.read_sql_query(q3, conn)

,gender,age,network_type
0,Female,15,3G
1,Female,17,4G


In [20]:
# Q4: Display the number of students for each internet type and network type.

q4 = """
SELECT l.internet_type, l.network_type, COUNT(s.student_id) AS student_count
FROM students s
INNER JOIN learning_environment l ON s.student_id = l.student_id
GROUP BY l.internet_type, l.network_type;
"""
pd.read_sql_query(q4, conn)

,internet_type,network_type,student_count
0,Mobile Data,3G,1
1,Mobile Data,4G,1
2,Wifi,3G,1
3,Wifi,4G,2


In [21]:
# Q5: Show education level, device, and number of students. 
# Include only students who have Poor OR Mid financial condition.

q5 = """
SELECT s.education_level, l.device, COUNT(s.student_id) AS student_count
FROM students s
INNER JOIN learning_environment l ON s.student_id = l.student_id
WHERE s.financial_condition IN ('Poor', 'Mid')
GROUP BY s.education_level, l.device;
"""
pd.read_sql_query(q5, conn)

,education_level,device,student_count
0,School,Phone,2
1,University,Laptop,2


In [22]:
# Q6: Display education level and the average class duration for each education level. 
# Only show education levels where the average class duration is greater than 1 hour.

q6 = """
SELECT s.education_level, AVG(l.class_duration) AS avg_duration
FROM students s
INNER JOIN learning_environment l ON s.student_id = l.student_id
GROUP BY s.education_level
HAVING AVG(l.class_duration) > 1;
"""
pd.read_sql_query(q6, conn)

,education_level,avg_duration
0,School,1.250000
1,University,2.333333


In [23]:
# Q7: Display device and the count of students. 
# Only include devices that are used by more than one student.

q7 = """
SELECT l.device, COUNT(s.student_id) AS student_count
FROM students s
INNER JOIN learning_environment l ON s.student_id = l.student_id
GROUP BY l.device
HAVING COUNT(s.student_id) > 1;
"""
pd.read_sql_query(q7, conn)

,device,student_count
0,Laptop,2
1,Phone,2


In [24]:
# Q8: Create a column "Internet_Quality" using CASE.
# If Internet Type is Wifi and Network Type is 4G -> 'Good', otherwise 'Limited'.

q8 = """
SELECT s.gender, s.education_level, l.internet_type, l.network_type,
       CASE 
           WHEN l.internet_type = 'Wifi' AND l.network_type = '4G' THEN 'Good'
           ELSE 'Limited'
       END AS Internet_Quality
FROM students s
INNER JOIN learning_environment l ON s.student_id = l.student_id;
"""
pd.read_sql_query(q8, conn)

,gender,education_level,internet_type,network_type,Internet_Quality
0,Male,University,Wifi,4G,Good
1,Female,School,Mobile Data,3G,Limited
2,Male,University,Wifi,4G,Good
3,Female,School,Mobile Data,4G,Limited
4,Male,University,Wifi,3G,Limited


In [26]:
# Q9: Create a VIEW called connected_students for Location = Yes AND Wifi.
# Then display records ordered by Adaptivity Level.

cursor.execute("DROP VIEW IF EXISTS connected_students")

cursor.execute("""
CREATE VIEW connected_students AS
SELECT s.student_id, s.gender, s.age, s.education_level, s.adaptivity_level, l.internet_type, l.location
FROM students s
INNER JOIN learning_environment l ON s.student_id = l.student_id
WHERE l.location = 'Yes' AND l.internet_type = 'Wifi';
""")

pd.read_sql_query("SELECT * FROM connected_students ORDER BY adaptivity_level", conn)

,student_id,gender,age,education_level,adaptivity_level,internet_type,location
0,3,Male,22,University,High,Wifi,Yes
1,1,Male,20,University,Moderate,Wifi,Yes


In [27]:
# Q10: Total students and Low adaptivity students per education level.
# Only show levels where more than 30% have Low adaptivity.

q10 = """
SELECT 
    s.education_level, 
    COUNT(s.student_id) AS total_students,
    SUM(CASE WHEN s.adaptivity_level = 'Low' THEN 1 ELSE 0 END) AS low_adaptivity_count
FROM students s
INNER JOIN learning_environment l ON s.student_id = l.student_id
GROUP BY s.education_level
HAVING (CAST(SUM(CASE WHEN s.adaptivity_level = 'Low' THEN 1 ELSE 0 END) AS FLOAT) / COUNT(s.student_id)) > 0.3;
"""

pd.read_sql_query(q10, conn)

,education_level,total_students,low_adaptivity_count
0,School,2,2
